# 06 — Chọn đặc trưng và cửa sổ thời gian

## I. Import thư viện

In [ ]:
# import os
# from google.colab import files

# os.makedirs('/content/data/processed', exist_ok=True)
# uploaded = files.upload()   # chọn 3 file: feature_data.csv, feature_groups.json, split_dates.csv
# for name in uploaded:
#     os.replace(name, f'/content/data/processed/{name}')
# print(os.listdir('/content/data/processed'))

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import random
import gc
import json
import os
from sklearn.preprocessing import StandardScaler
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = ROOT / 'data' / 'processed'
RESULT_DIR = ROOT / 'results'

print('GPU:', tf.config.list_physical_devices('GPU'))
print('Thư mục dữ liệu:', DATA_DIR)

## II. Đọc dữ liệu và lấy phần Train

In [ ]:
df = pd.read_csv(DATA_DIR / 'feature_data.csv', parse_dates=['Local Time'])
df = df.set_index('Local Time').sort_index()
splits = pd.read_csv(DATA_DIR / 'split_dates.csv', parse_dates=['Start', 'End'])
display(splits)


TRAIN_END = splits.loc[splits['Split'] == 'Train', 'End'].iloc[0]
VAL_START = splits.loc[splits['Split'] == 'Validation', 'Start'].iloc[0]
VAL_END = splits.loc[splits['Split'] == 'Validation', 'End'].iloc[0]
TEST_START = splits.loc[splits['Split'] == 'Test', 'Start'].iloc[0]

# target_real lấy TRƯỚC khi cắt df, vì bước chấm điểm cần cả chuỗi target của Train
target_real = df.loc[:TRAIN_END, 'PM25'].copy()
df = df.loc[:TRAIN_END].copy()

print('Số giờ Train:', len(df))
print('Target còn thiếu trong Train:', target_real.isna().sum())

## III. Thiết lập thí nghiệm

In [ ]:
HORIZONS = [3, 6, 12, 24]
WINDOWS = [24, 48, 72, 96]
SEEDS = [42, 123]
MAX_EPOCHS = 100   # EarlyStopping dừng sớm; 30 làm nhiều lần chạy bị cắt khi chưa học xong
PATIENCE = 5
BATCH_SIZE = 256
MIN_GAIN = 0.05

SAVE_DIR = str(RESULT_DIR / 'feature_selection')
os.makedirs(SAVE_DIR, exist_ok=True)
print('Thư mục lưu:', SAVE_DIR)

## IV. Danh sách feature ứng viên

In [ ]:
# Đọc nhóm feature từ 05_features, không chép tay để hai notebook không lệch nhau
with open(DATA_DIR / 'feature_groups.json', encoding='utf-8') as f:
    groups = json.load(f)

pollutants = groups['pollutants']
weather_base = groups['weather_base']
weather_derived = groups['weather_derived']
external = groups['external']
derived = groups['derived']
time_features = groups['time']

ALL_FEATURES = (['PM25'] + pollutants + weather_base + weather_derived
                + external + derived + time_features)

missing_cols = set(ALL_FEATURES) - set(df.columns)
if missing_cols:
    raise ValueError(f'Thiếu cột: {missing_cols}')

print('Tổng feature ứng viên:', len(ALL_FEATURES))
print('Trong đó: chất ô nhiễm', len(pollutants),
      '| khí tượng gốc', len(weather_base),
      '| khí tượng dẫn xuất', len(weather_derived),
      '| dữ liệu ngoài', len(external),
      '| dẫn xuất khác', len(derived),
      '| lịch', len(time_features))

## V. Tạo các mẫu chung để so sánh

In [ ]:
values = df[ALL_FEATURES].to_numpy(dtype='float32')
targets = target_real.to_numpy(dtype='float32')

In [ ]:
valid_row = np.isfinite(values).all(axis=1)

# Đòi đủ số giờ của window LỚN NHẤT, nếu không cửa sổ 96 giờ có thể chứa NaN
# và làm loss của GRU thành NaN. Mọi window dùng chung một tập mẫu nên so sánh vẫn công bằng.
CHECK_WINDOW = max(WINDOWS)

window_ok = (pd.Series(valid_row)
             .rolling(CHECK_WINDOW)
             .sum()
             .eq(CHECK_WINDOW)
             .to_numpy())

print(f'Giờ không NaN: {int(valid_row.sum())} / {len(valid_row)}')
print(f'Giờ có trọn {CHECK_WINDOW} giờ liên tục không NaN: {int(window_ok.sum())}')

In [ ]:
positions = np.arange(max(WINDOWS) - 1, len(df) - max(HORIZONS))
y_positions = positions[:, None] + np.asarray(HORIZONS)[None, :]
keep = window_ok[positions] & np.isfinite(targets[y_positions]).all(axis=1)
positions = positions[keep]
y_all = targets[positions[:, None] + np.asarray(HORIZONS)[None, :]]
times = df.index[positions]
last_target = times + pd.Timedelta(hours=max(HORIZONS))
assert last_target.max() <= TRAIN_END

print('Số mẫu giữ lại:', len(positions))

In [ ]:
print('Số mẫu chung:', len(times))
print('Input time:', times.min(), '→', times.max())
print('Target shape:', y_all.shape)

## VI. Chia ba fold theo thời gian

In [ ]:
span = TRAIN_END - df.index[0] + pd.Timedelta(hours=1)
val_width = pd.Timedelta(hours=int(span / pd.Timedelta(hours=1) * 0.15))
outer_start = TRAIN_END + pd.Timedelta(hours=1) - 3 * val_width

In [ ]:
folds = []
for k in range(3):
    start = outer_start + k * val_width
    end = start + val_width
    valid_indices = np.flatnonzero((times >= start) & (last_target < end))
    assert len(valid_indices), 'Fold validation rỗng.'
    train_indices = np.flatnonzero(last_target < times[valid_indices[0]])
    inner_boundary = times[train_indices[0]] + (times[train_indices[-1]] - times[train_indices[0]]) * 0.85
    inner_valid_indices = train_indices[times[train_indices] >= inner_boundary]
    inner_train_indices = train_indices[last_target[train_indices] < times[inner_valid_indices[0]]]
    assert len(inner_train_indices) > 100 and len(inner_valid_indices) > 100 and (len(valid_indices) > 100)
    assert last_target[inner_train_indices].max() < times[inner_valid_indices].min()
    assert last_target[train_indices].max() < times[valid_indices].min()
    folds.append({'fold': k + 1, 'train': train_indices, 'valid': valid_indices, 'inner_train': inner_train_indices, 'inner_valid': inner_valid_indices})

In [ ]:
fold_report = pd.DataFrame([{'Fold': f['fold'], 'Train_N': len(f['train']), 'Inner_train_N': len(f['inner_train']), 'Inner_val_N': len(f['inner_valid']), 'Outer_val_N': len(f['valid']), 'Train_last_target': last_target[f['train']].max(), 'Outer_first_issue': times[f['valid']].min(), 'Outer_last_target': last_target[f['valid']].max(), 'Inner_train_last_target': last_target[f['inner_train']].max(), 'Inner_first_issue': times[f['inner_valid']].min()} for f in folds])
display(fold_report)

## VII. Các hàm dùng khi thử mô hình

### 1. Cố định seed

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

### 2. Tạo GRU nhỏ

In [ ]:
def build_model(window, n_features, seed):
    tf.keras.backend.clear_session()
    set_seed(seed)
    model = tf.keras.Sequential([tf.keras.layers.Input(shape=(window, n_features)), tf.keras.layers.GRU(32), tf.keras.layers.Dropout(0.15), tf.keras.layers.Dense(32, activation='relu'), tf.keras.layers.Dense(len(HORIZONS))])
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001, clipnorm=1.0), loss='mae')
    return model

### 3. Lấy chuỗi đầu vào

In [ ]:
def get_X(features, window):
    feature_indices = [ALL_FEATURES.index(f) for f in features]
    return np.stack([values[p - window + 1:p + 1, feature_indices] for p in positions]).astype('float32')

### 4. Chuẩn hóa theo phần Train

In [ ]:
def scaling(X, y, train_idx, valid_idx):
    input_scaler, target_scaler = (StandardScaler(), StandardScaler())
    input_scaler.fit(X[train_idx].reshape(-1, X.shape[-1]))
    target_scaler.fit(y[train_idx].reshape(-1, 1))

    def scale_inputs(indices):
        shape = X[indices].shape
        return input_scaler.transform(X[indices].reshape(-1, shape[-1])).reshape(shape).astype('float32')

    def scale_targets(indices):
        return target_scaler.transform(y[indices].reshape(-1, 1)).reshape(-1, len(HORIZONS)).astype('float32')
    return (scale_inputs(train_idx), scale_targets(train_idx), scale_inputs(valid_idx), scale_targets(valid_idx), target_scaler)

### 5. Chọn epoch trên inner validation

In [ ]:
def choose_epochs(X, fold, window, feature_count, seed):
    X_fit_scaled, y_fit_scaled, X_eval_scaled, y_eval_scaled, target_scaler = scaling(X, y_all, fold['inner_train'], fold['inner_valid'])
    model = build_model(window, feature_count, seed)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE)
    history = model.fit(X_fit_scaled, y_fit_scaled, validation_data=(X_eval_scaled, y_eval_scaled), epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, shuffle=True, verbose=0, callbacks=[early_stop])
    best_epoch = int(np.argmin(history.history['val_loss']) + 1)
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return best_epoch

### 6. Học lại và dự báo outer validation

In [ ]:
def predict_outer_fold(X, fold, window, feature_count, seed, best_epoch):
    X_fit_scaled, y_fit_scaled, X_eval_scaled, y_eval_scaled, target_scaler = scaling(X, y_all, fold['train'], fold['valid'])
    model = build_model(window, feature_count, seed)
    model.fit(X_fit_scaled, y_fit_scaled, epochs=best_epoch, batch_size=BATCH_SIZE, shuffle=True, verbose=0)
    prediction_scaled = model.predict(X_eval_scaled, verbose=0)
    prediction = target_scaler.inverse_transform(prediction_scaled.reshape(-1, 1)).reshape(prediction_scaled.shape)
    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return prediction

### 7. Tính MAE và so với hai mốc đơn giản

In [ ]:
PM25_COL = ALL_FEATURES.index('PM25')

def score_outer_fold(prediction, fold, seed, best_epoch):
    validation_indices = fold['valid']
    pos = positions[validation_indices]
    actual = y_all[validation_indices]

    current_pm25 = values[pos, PM25_COL]
    persistence = np.repeat(current_pm25[:, None], len(HORIZONS), axis=1)
    # Cùng giờ hôm qua: vị trí pos + h - 24 luôn <= pos nên không dùng tương lai
    seasonal = targets[pos[:, None] + np.asarray(HORIZONS)[None, :] - 24]

    absolute_errors = np.abs(actual - prediction)
    mae_each_horizon = absolute_errors.mean(axis=0)
    last_train_target = last_target[fold['train']].max()
    threshold = float(target_real.loc[:last_train_target].quantile(0.9))
    high_mask = actual >= threshold
    high_mae = float(absolute_errors[high_mask].mean()) if high_mask.any() else None
    result = {'Fold': fold['fold'], 'Seed': seed, 'Epochs': best_epoch,
              'MAE': float(mae_each_horizon.mean()),
              'Persistence_MAE': float(np.abs(actual - persistence).mean()),
              'Seasonal_MAE': float(np.nanmean(np.abs(actual - seasonal))),
              # Trung bình của hai cách trên: mốc đơn giản khó vượt nhất ở bước kiểm tra dữ liệu
              'Blend_MAE': float(np.nanmean(np.abs(actual - (persistence + seasonal) / 2))),
              'High_PM_MAE': high_mae}
    for column, horizon in enumerate(HORIZONS):
        result[f'MAE_{horizon}h'] = float(mae_each_horizon[column])
        result[f'Persistence_MAE_{horizon}h'] = float(np.abs(actual[:, column] - persistence[:, column]).mean())
    return result

### 8. Thử một bộ feature và window, so sánh theo cặp

In [ ]:
experiment_rows = []
RUNS = {}

def evaluate(features, window, label):
    X = get_X(features, window)
    rows = []

    for fold in folds:
        for seed in SEEDS:
            print(label, '| window:', window, '| fold:', fold['fold'], '| seed:', seed)
            epochs = choose_epochs(X, fold, window, len(features), seed)
            prediction = predict_outer_fold(X, fold, window, len(features), seed, epochs)
            result = score_outer_fold(prediction, fold, seed, epochs)
            rows.append(result)
            print('MAE:', round(result['MAE'], 3), '| Epoch:', epochs)

    results = pd.DataFrame(rows)
    RUNS[label] = results
    score = results['MAE'].mean()
    experiment_rows.append({
        'Experiment': label,
        'Window': window,
        'N_features': len(features),
        'Features': '|'.join(features),
        'CV_MAE': score,
        'CV_MAE_std': results['MAE'].std(),
        'Persistence_MAE': results['Persistence_MAE'].mean(),
        'Seasonal_MAE': results['Seasonal_MAE'].mean(),
        'Blend_MAE': results['Blend_MAE'].mean(),
    })
    del X
    gc.collect()
    return score, results


def paired_diff(trial_runs, base_runs):
    """MAE(trial) - MAE(base) trên cùng (fold, seed). Âm = trial tốt hơn."""
    merged = trial_runs.merge(base_runs, on=['Fold', 'Seed'], suffixes=('_trial', '_base'))
    diff = merged['MAE_trial'] - merged['MAE_base']
    se = float(diff.std(ddof=1) / np.sqrt(len(diff))) if len(diff) > 1 else float('nan')
    return float(diff.mean()), se


def pick_simplest_within_1se(table, label_col, size_col):
    """Trong các lựa chọn cách cái tốt nhất không quá 1 SE (so theo cặp), chọn cái có size nhỏ nhất."""
    best_label = table.sort_values('CV_MAE').iloc[0][label_col]
    rows = []
    for _, row in table.iterrows():
        delta, se = paired_diff(RUNS[row['Run']], RUNS[table.set_index(label_col).loc[best_label, 'Run']])
        rows.append({label_col: row[label_col], 'Delta_vs_best': delta, 'SE': se,
                     'Within_1SE': delta <= (se if np.isfinite(se) else 0)})
    check = table.merge(pd.DataFrame(rows), on=label_col)
    chosen = check[check['Within_1SE']].sort_values([size_col, 'CV_MAE']).iloc[0]
    return chosen, check

## VIII. So sánh các nhóm feature

In [ ]:
feature_sets = {
    'A_PM25': ['PM25'],
    'B_PM25_PM10': ['PM25', 'PM10'],
    'C_pollutants': ['PM25'] + pollutants,
    'D_weather': ['PM25'] + weather_base,
    'E_pollutants_weather': ['PM25'] + pollutants + weather_base,
    'F_full_time': ['PM25'] + pollutants + weather_base + time_features,

    # Ba bộ dưới dùng để kiểm chứng dữ liệu mới có đóng góp hay không
    'G_derived_only': ['PM25'] + weather_derived + derived,
    'H_external_only': ['PM25'] + external,
    'I_extended_all': ALL_FEATURES.copy(),
}

print('Số bộ so sánh:', len(feature_sets))
for name, feats in feature_sets.items():
    print(f'  {name:<22} {len(feats):>2} feature')

In [ ]:
group_rows = []
for name, features in feature_sets.items():
    score, runs = evaluate(features, 48, name)
    group_rows.append({'Group': name, 'Run': name, 'CV_MAE': score,
                       'CV_MAE_std': runs['MAE'].std(), 'N_features': len(features)})
group_results = pd.DataFrame(group_rows).sort_values(['CV_MAE', 'N_features'])

# Hai mốc đơn giản không phụ thuộc bộ feature nên lấy từ lần chạy bất kỳ
any_runs = RUNS[group_results.iloc[0]['Run']]
print(f"Persistence MAE   : {any_runs['Persistence_MAE'].mean():.3f}")
print(f"Seasonal naive MAE: {any_runs['Seasonal_MAE'].mean():.3f}")
print(f"Trung bình 2 cách : {any_runs['Blend_MAE'].mean():.3f}  <- mô hình phải thấp hơn mốc này")

chosen_group, group_check = pick_simplest_within_1se(group_results, 'Group', 'N_features')
display(group_check.drop(columns='Run').sort_values('CV_MAE').round(3))

selected_features = feature_sets[chosen_group['Group']].copy()
current_runs = RUNS[chosen_group['Run']]
current_score = float(current_runs['MAE'].mean())
print('Bộ được chọn (gọn nhất trong 1 SE của bộ tốt nhất):', chosen_group['Group'])

## VIII.1 Đọc kết quả so sánh

In [ ]:
group_results.plot.barh(x='Group', y='CV_MAE', legend=False)
plt.xlabel('CV MAE (µg/m³)')
plt.title('So sánh nhóm đặc trưng')
plt.tight_layout()
plt.show()

## IX. Thử thêm các feature còn thiếu

In [ ]:
# Mỗi block là một nhóm được xét thêm/bỏ cùng nhau (cặp sin/cos phải đi cùng).
# Lấy từ ALL_FEATURES nên cột nào 05_features đã bỏ thì tự động không có ở đây.
PAIRS = [('wind_sin', 'wind_cos'), ('hour_sin', 'hour_cos'),
         ('dow_sin', 'dow_cos'), ('month_sin', 'month_cos')]
paired = {f for pair in PAIRS for f in pair}
feature_blocks = ([list(p) for p in PAIRS if all(f in ALL_FEATURES for f in p)]
                  + [[f] for f in ALL_FEATURES if f != 'PM25' and f not in paired])
print('Số block:', len(feature_blocks))

In [ ]:
add_rows = []

for block in feature_blocks:
    missing = [f for f in block if f not in selected_features]
    if not missing:
        continue

    trial = selected_features + missing
    label = 'ADD_' + '_'.join(missing)
    score, runs = evaluate(trial, 48, label)
    delta, se = paired_diff(runs, current_runs)
    add_rows.append({
        'Block': '|'.join(missing),
        'N_added': len(missing),
        'CV_MAE': score,
        'Gain': -delta,
        'SE': se,
        'Pass': (-delta >= MIN_GAIN) and (-delta > se),
    })

add_table = pd.DataFrame(add_rows).sort_values('CV_MAE')
display(add_table.head(10).round(3))

In [ ]:
# Xác nhận lần lượt tối đa 3 nhóm qua vòng lọc, mỗi lần so với bộ hiện tại (đã cập nhật)
for _, row in add_table[add_table['Pass']].head(3).iterrows():
    block = row['Block'].split('|')          # 'Block' là chuỗi nối bằng '|', phải tách lại
    to_add = [f for f in block if f not in selected_features]
    if not to_add:
        continue
    trial = selected_features + to_add
    score, runs = evaluate(trial, 48, 'CONFIRM_ADD_' + '_'.join(to_add))
    delta, se = paired_diff(runs, current_runs)
    print(f"{row['Block']}: gain {-delta:.3f}, SE {se:.3f}")
    if -delta >= MIN_GAIN and -delta > se:
        selected_features, current_runs, current_score = trial, runs, score
print('Features sau thêm:', selected_features)

## X. Chọn số giờ lịch sử

In [ ]:
window_rows = []
for w in WINDOWS:
    label = f'WINDOW_{w}'
    score, runs = evaluate(selected_features, w, label)
    window_rows.append({'Window': w, 'Run': label, 'CV_MAE': score, 'CV_MAE_std': runs['MAE'].std()})
window_results = pd.DataFrame(window_rows)

chosen_window, window_check = pick_simplest_within_1se(window_results, 'Window', 'Window')
display(window_check.drop(columns='Run').sort_values('Window').round(3))

best_window = int(chosen_window['Window'])
current_runs = RUNS[chosen_window['Run']]
current_score = float(current_runs['MAE'].mean())
print('Window được chọn (ngắn nhất trong 1 SE của window tốt nhất):', best_window)

In [ ]:
window_results.sort_values('Window').plot(x='Window', y='CV_MAE', marker='o', legend=False)
plt.xlabel('Số giờ lịch sử')
plt.ylabel('CV MAE (µg/m³)')
plt.tight_layout()
plt.show()

## XI. Thử bỏ các feature không giúp ích

In [ ]:
removal_rows = []
for block in feature_blocks:
    removed = [f for f in block if f in selected_features]
    if not removed:
        continue
    trial = [f for f in selected_features if f not in removed]
    if trial == ['PM25'] or not trial:
        continue
    old_score = current_score
    score, runs = evaluate(trial, best_window, 'REMOVE_' + '_'.join(removed))
    delta, se = paired_diff(runs, current_runs)
    # Bỏ nếu không xấu đi rõ rệt: tăng ít hơn MIN_GAIN và không vượt 1 SE
    accept = (delta < MIN_GAIN) and (delta <= se)
    if accept:
        selected_features, current_runs, current_score = trial, runs, score
    removal_rows.append({'Removed': '|'.join(removed), 'Before_MAE': old_score, 'Trial_MAE': score,
                         'Delta': delta, 'SE': se, 'Accepted': accept})
display(pd.DataFrame(removal_rows).round(3))

## XII. Tổng hợp và lưu cấu hình

In [ ]:
final_cv_mae = float(current_score)
final_cv_std = float(current_runs['MAE'].std())
persistence_mae = float(current_runs['Persistence_MAE'].mean())
seasonal_mae = float(current_runs['Seasonal_MAE'].mean())
blend_mae = float(current_runs['Blend_MAE'].mean())

print('Feature được chọn:', selected_features)
print('Window được chọn:', best_window)
print(f'CV MAE: {final_cv_mae:.3f} ± {final_cv_std:.3f} (độ lệch chuẩn giữa 6 lần chạy)')
print(f'Persistence MAE: {persistence_mae:.3f} | Seasonal naive MAE: {seasonal_mae:.3f} | Trung bình 2 cách: {blend_mae:.3f}')
print('Mô hình', 'TỐT HƠN' if final_cv_mae < blend_mae else 'CHƯA TỐT HƠN', 'mốc đơn giản nhất')
display(current_runs[['Fold', 'Seed'] + [f'MAE_{h}h' for h in HORIZONS]
                     + [f'Persistence_MAE_{h}h' for h in HORIZONS]].round(3))

pd.DataFrame(experiment_rows).to_csv(os.path.join(SAVE_DIR, 'experiments.csv'), index=False)

In [ ]:
config = {
    'selected_features': selected_features,
    'input_window': int(best_window),
    'horizons': HORIZONS,
    'target_column': 'PM25',
    'train_end': str(TRAIN_END),
    'validation_start': str(VAL_START),
    'validation_end': str(VAL_END),
    'test_start': str(TEST_START),

    'data_source': 'feature_data.csv',
    'total_candidates': len(ALL_FEATURES),
    'n_selected': len(selected_features),
    'selected_external': [f for f in selected_features if f in external],
    'selected_derived': [f for f in selected_features if f in derived],
    'selected_weather_derived': [f for f in selected_features if f in weather_derived],

    'selection_scope': 'CV trong Train (đến 31/08/2024)',
    'selection_cv_mae': final_cv_mae,
    'selection_cv_mae_std': final_cv_std,
    'persistence_cv_mae': persistence_mae,
    'seasonal_naive_cv_mae': seasonal_mae,
    'blend_baseline_cv_mae': blend_mae,
    'selection_rule': 'paired fold x seed; add if gain >= MIN_GAIN and > 1 SE; '
                      'remove if loss < MIN_GAIN and <= 1 SE; group/window = simplest within 1 SE',
    'min_gain': MIN_GAIN,
    'seeds': SEEDS,

    'test_used_for_selection': False,
    'test_previously_explored_in_eda': True,
    'validation_previously_used_for_selection': True,
}

config_path = os.path.join(SAVE_DIR, 'feature_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print('Đã lưu:', config_path)
print('Feature được chọn:', len(selected_features), '/', len(ALL_FEATURES))
print('Từ dữ liệu ngoài     :', config['selected_external'])
print('Từ dẫn xuất khí tượng:', config['selected_weather_derived'])
print('Từ dẫn xuất khác     :', config['selected_derived'])

In [ ]:
# !zip -r results.zip results
# files.download('results.zip')